# 1D viscous Burgers: interactive space–time audit

The periodic benchmark solves

$$u_t+u u_x=\nu u_{xx},\qquad x\in[0,2\pi),\qquad \nu=10^{-2}.$$

A smooth multi-mode initial condition steepens into a resolved viscous shock. The panels mirror the 1D KS notebook so distribution, spectrum, topology, and PDE diagnostics can be compared directly.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

case_name = "viscous_burgers"
smoke_mode = os.environ.get("PHYCOFLOW_VIZ_SMOKE", "0") == "1"
n = 64 if smoke_mode else 512
domain_length = 2.0 * np.pi
viscosity = 1.0e-2
solver_dt = 0.002 if smoke_mode else 0.0025
final_time = 0.02 if smoke_mode else 2.0
save_every = 1 if smoke_mode else 4


In [2]:
x = np.linspace(0.0, domain_length, n, endpoint=False)
dx = domain_length / n
wave = 2.0 * np.pi * np.fft.fftfreq(n, d=dx)
modes = np.fft.fftfreq(n) * n
dealias = np.abs(modes) <= n / 3
linear = -viscosity * wave**2

def nonlinear_term(u_hat):
    u = np.fft.ifft(u_hat).real
    return -0.5j * wave * np.fft.fft(u**2) * dealias

u0 = -np.sin(x) + 0.25 * np.sin(2.0 * x + 0.3) + 0.10 * np.sin(3.0 * x - 0.2)
roots = np.exp(1j * np.pi * (np.arange(1, 17) - 0.5) / 16)
lr = solver_dt * linear[:, None] + roots
E = np.exp(solver_dt * linear)
E2 = np.exp(0.5 * solver_dt * linear)
Q = solver_dt * np.real(np.mean((np.exp(lr / 2.0) - 1.0) / lr, axis=-1))
f1 = solver_dt * np.real(np.mean((-4.0 - lr + np.exp(lr) * (4.0 - 3.0 * lr + lr**2)) / lr**3, axis=-1))
f2 = solver_dt * np.real(np.mean((2.0 + lr + np.exp(lr) * (-2.0 + lr)) / lr**3, axis=-1))
f3 = solver_dt * np.real(np.mean((-4.0 - 3.0 * lr - lr**2 + np.exp(lr) * (4.0 - lr)) / lr**3, axis=-1))

u_hat = np.fft.fft(u0) * dealias
snapshots, times_list = [], []
n_steps = int(round(final_time / solver_dt))
for step in range(n_steps + 1):
    if step % save_every == 0 or step == n_steps:
        snapshots.append(np.fft.ifft(u_hat).real)
        times_list.append(step * solver_dt)
    if step == n_steps:
        break
    Nv = nonlinear_term(u_hat)
    a = E2 * u_hat + Q * Nv
    Na = nonlinear_term(a)
    b = E2 * u_hat + Q * Na
    Nb = nonlinear_term(b)
    c = E2 * a + Q * (2.0 * Nb - Nv)
    Nc = nonlinear_term(c)
    u_hat = (E * u_hat + f1 * Nv + 2.0 * f2 * (Na + Nb) + f3 * Nc) * dealias

times = np.asarray(times_list)
fields = np.asarray(snapshots)[:, None, :]  # [time, channel, x]


In [3]:
def spatial_pde_terms(field):
    field_hat = np.fft.fft(field)
    advection = np.fft.ifft(0.5j * wave * np.fft.fft(field**2) * dealias).real
    diffusion = np.fft.ifft(-viscosity * wave**2 * field_hat).real
    return advection - diffusion

residuals, time_terms = [], []
for index in range(1, times.size - 1):
    ut = (fields[index + 1, 0] - fields[index - 1, 0]) / (times[index + 1] - times[index - 1])
    residuals.append(ut + spatial_pde_terms(fields[index, 0]))
    time_terms.append(ut)
gradient_max = max(np.max(np.abs(np.fft.ifft(1j * wave * np.fft.fft(frame)).real)) for frame in fields[:, 0])
diagnostics = {
    "relative_pde_residual": float(np.linalg.norm(residuals) / (np.linalg.norm(time_terms) + 1.0e-12)),
    "finite": bool(np.isfinite(fields).all()),
    "maximum_abs_gradient": float(gradient_max),
    "mean_drift": float(np.max(np.abs(fields[:, 0].mean(axis=-1) - fields[0, 0].mean()))),
}
diagnostics


{'relative_pde_residual': 0.005050546188209093,
 'finite': True,
 'maximum_abs_gradient': 56.859032210654476,
 'mean_drift': 1.1102230246251565e-16}

In [4]:
def draw_frame(frame_index, axes):
    for axis in axes.ravel():
        axis.clear()
    field = fields[frame_index, 0]
    axes[0, 0].imshow(fields[:, 0], origin="lower", aspect="auto", extent=(0, domain_length, times[0], times[-1]), cmap="RdBu_r")
    axes[0, 0].axhline(times[frame_index], color="black", lw=1)
    axes[0, 0].set(xlabel="x", ylabel="t", title="Shock-forming space–time field")
    axes[0, 1].plot(x, field, color="tab:blue")
    axes[0, 1].set(xlim=(0, domain_length), title=f"u(x), t={times[frame_index]:.2f}")
    axes[0, 2].hist(field, bins=30, density=True, color="tab:blue", alpha=0.8)
    axes[0, 2].set_title("Value PDF (Wasserstein input)")
    power = np.abs(np.fft.rfft(field - field.mean()))**2 / field.size**2
    axes[1, 0].semilogy(np.arange(power.size)[1:], power[1:] + 1.0e-18, color="tab:orange")
    axes[1, 0].set_title("1D power spectrum")
    threshold = np.median(field)
    axes[1, 1].fill_between(x, 0, field > threshold, step="mid", alpha=0.7)
    axes[1, 1].set(xlim=(0, domain_length), ylim=(0, 1.1), title="Superlevel components")
    ux = np.fft.ifft(1j * wave * np.fft.fft(field)).real
    axes[1, 2].plot(x, ux, color="tab:red")
    axes[1, 2].set(xlim=(0, domain_length), title="Shock gradient $u_x$")
    return []

dashboard = None
if not smoke_mode:
    frame_indices = np.unique(np.linspace(0, times.size - 1, min(times.size, 31), dtype=int))
    figure, dashboard_axes = plt.subplots(2, 3, figsize=(12, 7), dpi=72, constrained_layout=True)
    animation = FuncAnimation(figure, lambda i: draw_frame(i, dashboard_axes), frames=frame_indices, interval=180, repeat=True)
    plt.close(figure)
    dashboard = HTML(animation.to_jshtml(default_mode="once"))
    display(dashboard)
